# SQL Validation: A/B Test Funnel & Conversion Analysis

This notebook complements the main analysis (`AB_Test_Analysis_Ecommerce.ipynb`) by replicating the core funnel and conversion metrics using pure SQL against a relational database, instead of pandas DataFrames.

**Why this matters:** the original analysis worked with data already merged into DataFrames. Here, the four raw source files are loaded into a SQLite database as separate related tables, and all aggregation, filtering, and joins are done directly in SQL — closer to how this analysis would run against a production database.

**Data files used** (same raw sources as the main analysis):
- `final_ab_events_upd_us.csv` — user event log (login, product_page, product_cart, purchase)
- `final_ab_new_users_upd_us.csv` — new user registrations (region, device)
- `final_ab_participants_upd_us.csv` — test/group assignment per user
- `ab_project_marketing_events_us.csv` — marketing campaign calendar

## 1. Load data into a relational database

Each CSV becomes its own SQL table, connected by `user_id`. This mirrors a real production setup, where this data would live in separate tables rather than a single flat file.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ab_test_ecommerce.db')

events = pd.read_csv('../data/final_ab_events_upd_us.csv')
users = pd.read_csv('../data/final_ab_new_users_upd_us.csv')
participants = pd.read_csv('../data/final_ab_participants_upd_us.csv')
marketing = pd.read_csv('../data/ab_project_marketing_events_us.csv')

events.to_sql('events', conn, if_exists='replace', index=False)
users.to_sql('users', conn, if_exists='replace', index=False)
participants.to_sql('participants', conn, if_exists='replace', index=False)
marketing.to_sql('marketing_events', conn, if_exists='replace', index=False)
conn.commit()

print(f"events: {len(events):,} rows | users: {len(users):,} rows | participants: {len(participants):,} rows")

events: 423,761 rows | users: 58,703 rows | participants: 14,525 rows


## 2. Data quality check: overlapping test assignment

Before trusting any group comparison, it's worth checking whether any user was assigned to more than one experiment at the same time — this would contaminate the results, since their behavior couldn't be attributed to a single test.

In [2]:
query = '''
SELECT user_id, COUNT(DISTINCT ab_test) AS num_tests
FROM participants
GROUP BY user_id
HAVING num_tests > 1
'''
dual_test_users = pd.read_sql(query, conn)
print(f"Users assigned to both tests simultaneously: {len(dual_test_users)}")

Users assigned to both tests simultaneously: 887


**Finding:** 887 users were enrolled in both `recommender_system_test` and `interface_eu_test` at the same time. This is a data quality issue worth flagging — any conclusion drawn from either test individually should note that a subset of participants were exposed to two simultaneous experiments, which could confound the results. This kind of check is a standard first step before trusting any A/B test output, and is easy to express in SQL with a `GROUP BY` + `HAVING COUNT(DISTINCT ...)`.

## 3. Funnel by group (`recommender_system_test`)

Replicating the core funnel metric from the main analysis: unique users reaching each stage (`login`, `product_page`, `product_cart`, `purchase`), split by test group.

In [3]:
query = '''
SELECT participants."group", events.event_name, COUNT(DISTINCT events.user_id) AS unique_users
FROM participants
JOIN events ON participants.user_id = events.user_id
WHERE participants.ab_test = 'recommender_system_test'
GROUP BY participants."group", events.event_name
ORDER BY participants."group", unique_users DESC
'''
pd.read_sql(query, conn)

,group,event_name,unique_users
0,A,login,2747
1,A,product_page,1780
2,A,purchase,872
3,A,product_cart,824
4,B,login,927
5,B,product_page,523
6,B,purchase,256
7,B,product_cart,255


One thing worth calling out just from this table: group A has roughly 3x more users than group B (2,747 vs 928 at the login stage). That imbalance matters for interpreting any group comparison — it's covered in more detail in the main analysis's statistical testing section, but it's visible here directly from the raw counts.

## 4. Conversion rate by group

The main result from the original analysis: what share of users in each group completed a `purchase`. Built here using two CTEs — one for total users per group, one for users who purchased — joined together.

In [4]:
query = '''
WITH total_users AS (
    SELECT "group", COUNT(DISTINCT user_id) AS total_users
    FROM participants
    WHERE ab_test = 'recommender_system_test'
    GROUP BY "group"
),
purchasers AS (
    SELECT participants."group", COUNT(DISTINCT events.user_id) AS purchasers
    FROM participants
    JOIN events ON participants.user_id = events.user_id
    WHERE participants.ab_test = 'recommender_system_test'
      AND events.event_name = 'purchase'
    GROUP BY participants."group"
)
SELECT 
    total_users."group",
    total_users.total_users,
    purchasers.purchasers,
    ROUND(100.0 * purchasers.purchasers / total_users.total_users, 2) AS conversion_rate_pct
FROM total_users
JOIN purchasers ON total_users."group" = purchasers."group"
'''
pd.read_sql(query, conn)

,group,total_users,purchasers,conversion_rate_pct
0,A,2747,872,31.74
1,B,928,256,27.59


## 5. Marketing calendar overlap check

The `marketing_events` table wasn't used in the main analysis. It's worth checking whether the test period overlapped with any marketing campaign — a promo running during the test would be a second confounding factor, alongside the dual-test enrollment issue found above.

In [5]:
query = '''
SELECT name, regions, start_dt, finish_dt
FROM marketing_events
WHERE start_dt <= '2020-12-30' AND finish_dt >= '2020-12-07'
'''
pd.read_sql(query, conn)

,name,regions,start_dt,finish_dt
0,Christmas&New Year Promo,"EU, N.America",2020-12-25,2021-01-03
1,CIS New Year Gift Lottery,CIS,2020-12-30,2021-01-07


**Finding:** the `recommender_system_test` ran from Dec 7 to Dec 30, 2020, which overlaps directly with the **Christmas & New Year Promo** (Dec 25 – Jan 3, active in EU and N. America). This is a real confound: any lift in purchases during the last week of the test could be driven by the holiday promotion rather than the recommender system being tested. This is the kind of check that's easy to miss when working from an already-merged DataFrame, but becomes a one-line SQL query once the marketing calendar is loaded as its own table.

## 6. Conversion rate by device

Breaking the group conversion rates down by `device` (from the `users` table) to check whether the A/B effect is consistent across platforms, or concentrated in one device type.

In [6]:
query = '''
WITH total AS (
    SELECT users.device, participants."group", COUNT(DISTINCT participants.user_id) AS total_users
    FROM participants
    JOIN users ON participants.user_id = users.user_id
    WHERE participants.ab_test = 'recommender_system_test'
    GROUP BY users.device, participants."group"
),
buyers AS (
    SELECT users.device, participants."group", COUNT(DISTINCT events.user_id) AS buyers
    FROM participants
    JOIN users ON participants.user_id = users.user_id
    JOIN events ON participants.user_id = events.user_id
    WHERE participants.ab_test = 'recommender_system_test' AND events.event_name = 'purchase'
    GROUP BY users.device, participants."group"
)
SELECT total.device, total."group", total.total_users, buyers.buyers,
       ROUND(100.0 * buyers.buyers / total.total_users, 2) AS conversion_rate_pct
FROM total
JOIN buyers ON total.device = buyers.device AND total."group" = buyers."group"
ORDER BY total.device, total."group"
'''
pd.read_sql(query, conn)

,device,group,total_users,buyers,conversion_rate_pct
0,Android,A,1197,375,31.33
1,Android,B,428,120,28.04
2,Mac,A,270,97,35.93
3,Mac,B,76,23,30.26
4,PC,A,726,220,30.30
5,PC,B,227,59,25.99
6,iPhone,A,554,180,32.49
7,iPhone,B,197,54,27.41


**Finding:** Group A outperforms Group B on every device (Mac: 35.9% vs 30.3%, iPhone: 32.5% vs 27.4%, Android: 31.3% vs 28.0%, PC: 30.3% vs 26.0%). The gap is fairly consistent across platforms — Mac shows the highest conversion for both groups, but there's no single device driving the overall A-vs-B difference. This rules out "the result is just a device-mix artifact" as an explanation for the gap.

## 7. Time from registration to first purchase

A different angle on the same question: not just *whether* users convert, but *how fast*. This uses SQLite's `JULIANDAY()` function to compute the number of days between `first_date` (registration) and each user's first `purchase` event.

In [7]:
query = '''
WITH first_purchase AS (
    SELECT user_id, MIN(event_dt) AS first_purchase_dt
    FROM events
    WHERE event_name = 'purchase'
    GROUP BY user_id
)
SELECT 
    participants."group",
    ROUND(AVG(JULIANDAY(first_purchase.first_purchase_dt) - JULIANDAY(users.first_date)), 2) AS avg_days_to_purchase
FROM participants
JOIN users ON participants.user_id = users.user_id
JOIN first_purchase ON participants.user_id = first_purchase.user_id
WHERE participants.ab_test = 'recommender_system_test'
GROUP BY participants."group"
'''
pd.read_sql(query, conn)

,group,avg_days_to_purchase
0,A,0.50
1,B,0.54


**Finding:** both groups convert at essentially the same speed once they do convert (Group A: 0.50 days, Group B: 0.54 days on average). So the recommender system doesn't appear to speed up the decision to purchase — the difference between groups shows up in *whether* a user buys at all (the conversion rate gap), not in *how quickly* they buy once they're going to.

## Conclusion

The SQL-based conversion rates (Group A: 31.74%, Group B: 27.59%) match the results from the pandas-based analysis in the main notebook, confirming the funnel logic is consistent across both approaches.

This validation step also surfaced findings not covered in the original notebook:
- **887 users** enrolled in both simultaneous experiments — a data quality concern for either test's results.
- The test period **overlapped with the Christmas & New Year marketing promo**, a second potential confound affecting the last week of the test.
- The Group A vs. B conversion gap **holds consistently across all four device types**, ruling out device mix as the driver.
- Both groups convert at a **similar speed** once they do purchase — the effect is on conversion likelihood, not conversion speed.

**Takeaway:** the same business question can be answered equally well against raw relational tables using SQL (JOINs, GROUP BY, HAVING, CTEs, date functions) as it can with a pre-merged pandas DataFrame — and working from the raw, separated tables surfaced two confounding factors (dual-test enrollment, campaign overlap) that are easy to miss once everything is already flattened into a single DataFrame.